In [ ]:
#loads libraries
import numpy as np
import pandas as pd

from transformers import pipeline
from transformers import AutoTokenizer
from transformers import default_data_collator
from transformers import AutoModelForSeq2SeqLM, AutoModelForCausalLM
from transformers import get_linear_schedule_with_warmup

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from peft import IA3Config, get_peft_model
from peft import PeftModel, PeftConfig
from datasets import Dataset

import gc
import time

In [ ]:
#loads tabular data into numpy array
df = pd.read_csv('AI_test.csv', sep=';')
print(df)

columns = df.columns.values
print(columns)

In [ ]:
#turn one percent of the data into labeled examples
#we want exactly 500 positive and 500 negative examples
labeled_examples = []
status = ""
pos = 500
neg = 500
index = -1
while len(labeled_examples)!=1000:
    index += 1
    sentence= ""
    #if rating > 3 it's positive, otherwise it's negativee
    if float(df.iloc[index]['rating']) > 3:
        #if we already have 500 positive examples, we don't create more
        if pos == 0:
          continue
        status = "Likes"
        pos -= 1
    else:
        if neg == 0:
          continue
        status = "Dislikes"
        neg -= 1
    #create sentences describing the values taken by the columns for each row
    for i in range(len(columns)):
      sentence += f"The {columns[i]} is {df.iloc[index,i]}."
    labeled_examples.append((sentence,status,float(df.iloc[index,2])))
    df.drop(index, inplace=True)

#fix the indices in the original df
df = df.reset_index(drop=True)
print(df)

# **TabLLM**

In [ ]:
#create sentences for the rest of the df
sentences = []
for m in range(len(df)):
  sentence= ""
  for i in range(len(columns)):
    sentence += f"The {columns[i]} is {df.iloc[m,i]}."
  sentences.append(sentence)

#replace the existing columns with the sentences
df['Sentence'] = sentences
#df = df.drop(['userId','movieId','rating','timestamp'], axis = 1)
print(df)

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")
#gpt2 | openai-community/gpt2-xl | Qwen/Qwen2.5-3B-Instruct | microsoft/Phi-3-mini-4k-instruct | mistralai/Mistral-7B-v0.3

#*Zero-Shot Prompting*

In [ ]:
#build the prompt, include the question as well as known positive and negative examples
prompt = "Classify if the user likes or dislikes the movie based on the description. \n\n "

#for each iteration of the loop, feed it the examples as well as the one you want an answer to
for i in range(100):
  test_sentence = df.iloc[i]['Sentence']
  final_prompt = prompt + f"Description: {test_sentence}\nVerdict:"
  answer = generator(final_prompt, return_full_text=False)
  print(test_sentence)
  print(answer[0]['generated_text'])
  print("----------------------------------------------------------------------------------")

In [ ]:
#resource evaluation
prompt = (
    "Classify if the user likes or dislikes the movie based on the description.\n\n "
)
sample_queries = [
    f"{prompt} Description: {df.iloc[i]['Sentence']}\nVerdict:"
    for i in range(5)
]


class ComputeProfiler:

  def __init__(self, tokenizer):
    self.tokenizer = tokenizer
    self.latencies = []
    self.input_tokens = []
    self.output_tokens = []
    self.peak_vrams = []

  def profile_query(self, query_fn, *args, **kwargs):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()
    output_text, input_prompt = query_fn(*args, **kwargs)
    end_time = time.perf_counter()

    self.latencies.append(end_time - start_time)
    self.peak_vrams.append(torch.cuda.max_memory_allocated() / (1024**3))

    self.input_tokens.append(len(self.tokenizer.encode(input_prompt)))
    self.output_tokens.append(len(self.tokenizer.encode(output_text)))

    return output_text

  def summary(self, model_name: str, method_name: str):
    return {
        "Paradigm / Method": method_name,
        "Model": model_name,
        "Avg Latency (s)": (
            f"{np.mean(self.latencies):.2f} ± {np.std(self.latencies):.2f}"
        ),
        "Avg Input Tokens": f"{int(np.mean(self.input_tokens))}",
        "Avg Output Tokens": f"{int(np.mean(self.output_tokens))}",
        "Peak VRAM (GB)": f"{np.max(self.peak_vrams):.2f}",
    }

profiler = ComputeProfiler(generator.tokenizer)

def run_qwen_inference(q):
  pad_id = (
      generator.tokenizer.pad_token_id
      if generator.tokenizer.pad_token_id is not None
      else generator.tokenizer.eos_token_id
  )

  res = generator(
      q,
      max_new_tokens=64,
      return_full_text=False,
      pad_token_id=pad_id,
      do_sample=False,
  )

  generated_entry = res[0]["generated_text"]

  if isinstance(generated_entry, list):
    out_text = generated_entry[-1]["content"].strip()
  else:
    out_text = str(generated_entry).strip()

  return out_text, q

for s in sample_queries:
  profiler.profile_query(run_qwen_inference, s)

print(profiler.summary("Qwen2.5-3B-Instruct (4-bit)", "Zero Shot"))

# *In-Context Learning*

In [ ]:
#build the prompt, include the question as well as known positive and negative examples
prompt = "Classify if the user likes or dislikes the movie based on the description. \n\n "

i, j = 5, 5
for k in range(len(labeled_examples)):
  if labeled_examples[k][1] == "Likes" and i > 0:
    prompt += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"
    i -= 1
  elif labeled_examples[k][1] == "Dislikes" and j > 0:
    prompt += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"
    j -= 1
  if i == 0 and j == 0:
    break

#for each iteration of the loop, feed it the examples as well as the one you want an answer to
for i in range(100):
  test_sentence = df.iloc[i]['Sentence']
  final_prompt = prompt + f"Description: {test_sentence}\nVerdict:"
  answer = generator(final_prompt, max_new_tokens=2, return_full_text=False)
  print(test_sentence)
  print(answer[0]['generated_text'])
  print("----------------------------------------------------------------------------------")

# *T-Few*

In [ ]:
#T-Few method for determining an answer
labeled_df = pd.DataFrame(labeled_examples)
labeled_df.rename(columns={0: 'Sentence', 1: 'Label', 2: 'drop'}, inplace=True)
labeled_df = labeled_df.drop('drop',axis=1)
unlabeled_df = df
print(labeled_df)
print(unlabeled_df)

text_column = 0
label_column = 1

tokenizer = AutoTokenizer.from_pretrained("bigscience/mt0-large")

def preprocess_function(examples):
    model_inputs = tokenizer(examples[text_column], max_length=128, padding="max_length", truncation=True)
    labels = tokenizer(examples[label_column], max_length=3, padding="max_length", truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

labeled_df = labeled_df.map(preprocess_function)

train_df = labeled_df["Sentence"]
eval_df = labeled_df["Label"]

train_dataloader = DataLoader(train_df, shuffle=True, collate_fn=default_data_collator, batch_size=8, pin_memory=True)
eval_dataloader = DataLoader(eval_df, collate_fn=default_data_collator, batch_size=8, pin_memory=True)

model = AutoModelForSeq2SeqLM.from_pretrained("bigscience/mt0-large")

peft_config = IA3Config(task_type="SEQ_2_SEQ_LM")
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
num_epochs = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

model.save_pretrained("./my_ia3_model")
tokenizer.save_pretrained("./my_ia3_model")

In [ ]:
#load model and set it to evaluation mode
#evaluating with the base model did not yield better results
base_model = AutoModelForSeq2SeqLM.from_pretrained("bigscience/mt0-large")
model = PeftModel.from_pretrained(base_model, "./my_ia3_model")
model.to(device)
model.eval()

#test for 100 examples
for i in range(100):
  input = unlabeled_df.iloc[i].values[4] + f" Question: Do you think the user Likes or Dislikes the movie? Answer: "
  print(input)
  input = tokenizer(input, return_tensors="pt").to(device)

  with torch.no_grad():
      input = {k: v.to(device) for k, v in input.items()}
      outputs = model.generate(input_ids=input["input_ids"], max_new_tokens=10)
      prediction = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)
      print(prediction)

# *T-Few + Text Modification Part 1*

In [ ]:
#it's possible that the reason for the above results is because the sentences are too complex, so we'll simplify them and retrain it
for i in range(len(labeled_examples)):
  labeled_examples[i] = (f"The rating is: {labeled_examples[i][2]}/5.0",labeled_examples[i][1])

#modify the format of the sentences on the test dataframe as well
for j in range(len(df)):
  df.iloc[j,2] = f"The rating is: {float(df.iloc[j,2])}/5.0"

print(labeled_examples[0])
print(df)

In [ ]:
#same process as above
labeled_df = pd.DataFrame(labeled_examples)
labeled_df.rename(columns={0: 'Sentence', 1: 'Label'}, inplace=True)
unlabeled_df = df['rating']
print(labeled_df)
print(unlabeled_df)

text_column = 0
label_column = 1

tokenizer = AutoTokenizer.from_pretrained("bigscience/mt0-large")

def preprocess_function(examples):
    model_inputs = tokenizer(examples[text_column], max_length=128, padding="max_length", truncation=True)
    labels = tokenizer(examples[label_column], max_length=3, padding="max_length", truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

labeled_df = labeled_df.map(preprocess_function)

train_df = labeled_df["Sentence"]
eval_df = labeled_df["Label"]

train_dataloader = DataLoader(train_df, shuffle=True, collate_fn=default_data_collator, batch_size=8, pin_memory=True)
eval_dataloader = DataLoader(eval_df, collate_fn=default_data_collator, batch_size=8, pin_memory=True)

model = AutoModelForSeq2SeqLM.from_pretrained("bigscience/mt0-large")

peft_config = IA3Config(task_type="SEQ_2_SEQ_LM")
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
num_epochs = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

model.save_pretrained("./my_ia3_model")
tokenizer.save_pretrained("./my_ia3_model")

In [ ]:
base_model = AutoModelForSeq2SeqLM.from_pretrained("bigscience/mt0-large")
model = PeftModel.from_pretrained(base_model, "./my_ia3_model")
model.to(device)
model.eval()

for i in range(100):
  input = unlabeled_df.iloc[i] + f" Does the user like or dislike this movie?"
  print(input)
  input = tokenizer(input, return_tensors="pt").to(device)

  with torch.no_grad():
      input = {k: v.to(device) for k, v in input.items()}
      outputs = model.generate(input_ids=input["input_ids"], max_new_tokens=5)
      prediction = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)
      print(prediction)

# *T-Few + Text Modification Part 2*

In [ ]:
labeled_df = pd.DataFrame(labeled_examples)
labeled_df.rename(columns={0: 'Sentence', 1: 'Label'}, inplace=True)
unlabeled_df = df
print(labeled_df)
print(unlabeled_df)

text_column = "Sentence"
label_column = "Label"

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    prompt = f"Data: {examples[text_column]} Verdict:"
    answer = f" {examples[label_column]}"

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]

    input_ids = prompt_ids + answer_ids

    labels = [-100] * len(prompt_ids) + answer_ids

    padding_len = 128 - len(input_ids)
    input_ids += [tokenizer.pad_token_id] * padding_len
    labels += [-100] * padding_len

    return {
        "input_ids": torch.tensor(input_ids[:128]),
        "labels": torch.tensor(labels[:128]),
        "attention_mask": torch.tensor(([1] * len(input_ids[:128])))
    }

full_dataset = Dataset.from_pandas(labeled_df)
split_dataset = full_dataset.train_test_split(test_size=0.2)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

tokenized_train = train_dataset.map(preprocess_function, batched=False)
tokenized_eval = eval_dataset.map(preprocess_function, batched=False)

tokenized_train = tokenized_train.remove_columns(["Sentence", "Label"])
tokenized_eval = tokenized_eval.remove_columns(["Sentence", "Label"])

train_dataloader = DataLoader(tokenized_train, shuffle=True, collate_fn=default_data_collator, batch_size=8)
eval_dataloader = DataLoader(tokenized_eval, collate_fn=default_data_collator, batch_size=8)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

peft_config = IA3Config(task_type="CAUSAL_LM")
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
num_epochs = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

model.save_pretrained("./my_ia3_model")
tokenizer.save_pretrained("./my_ia3_model")

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = PeftModel.from_pretrained(base_model, "./my_ia3_model")
model.to(device)
model.eval()

for i in range(100):
  input = unlabeled_df.iloc[i] + f" Does the user like or dislike this movie?"
  input = tokenizer(input, return_tensors="pt").to(device)

  with torch.no_grad():
      input = {k: v.to(device) for k, v in input.items()}
      outputs = model.generate(input_ids=input["input_ids"], max_new_tokens=2)
      prediction = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)
      print(prediction)